# 02 · Error analysis

Where does the champion model go wrong, and what does that mean for a fraud-ops team?
* **False positives**: high scores on legitimate customers. This is where friction lands.
* **Missed fraud**: fraud the recommended policy approved. Card testing? A product line? New accounts?

In [1]:
import os
import altair as alt
import numpy as np
import pandas as pd

from fraud import charts, db  # charts registers the project's Altair theme
from fraud.config import get_paths
from fraud.io import load_json

PROFILE = os.environ.get("FRAUD_PROFILE", "full")  # "sample" = synthetic data
paths = get_paths(PROFILE)
pd.set_option("display.max_columns", 50, "display.width", 160)
print("profile:", PROFILE)

profile: full


In [2]:
res = load_json(paths.reports / "train_results.json")
champ = res["champion"]
scored = pd.read_parquet(paths.scored)
test = scored[scored.split == "test"].copy()
dec = pd.read_parquet(paths.processed / "decisions.parquet")
test = test.merge(dec, on="TransactionID")
p = f"p_{champ}"
test["amount_band"] = pd.cut(test.TransactionAmt, [0, 10, 50, 200, 1000, 1e9], labels=["<$10", "$10-50", "$50-200", "$200-1k", "$1k+"])
print(champ, len(test), "test rows")

lgbm_tuned 88581 test rows


## 1. Top 20 false positives

In [3]:
cols = ["TransactionID", p, "TransactionAmt", "ProductCD", "has_identity", "is_first_txn_uid", "card_tx_1h",
        "amt_vs_uid_mean", "email_match", "device_family", "p_email_provider", "decision"]
fp = test[test.isFraud == 0].nlargest(20, p)[cols]
fp

,TransactionID,p_lgbm_tuned,TransactionAmt,ProductCD,has_identity,is_first_txn_uid,card_tx_1h,amt_vs_uid_mean,email_match,device_family,p_email_provider,decision
80686,3569645,0.993398,130.645,C,1,1.0,0,NaN,1.0,NaN,google,decline
80383,3569342,0.990710,78.387,C,1,0.0,0,2.119685,1.0,NaN,google,decline
79859,3568818,0.989106,22.625,C,1,0.0,0,0.248701,1.0,NaN,microsoft,decline
80790,3569749,0.986860,128.001,C,1,1.0,0,NaN,1.0,gecko browser string,google,decline
80740,3569699,0.986120,130.645,C,1,1.0,0,NaN,1.0,NaN,google,decline
34906,3523865,0.984868,20.903,C,1,0.0,2,0.638131,1.0,NaN,google,decline
79857,3568816,0.982985,22.625,C,1,0.0,0,0.279004,1.0,NaN,microsoft,decline
34911,3523870,0.981626,20.903,C,1,0.0,3,1.076561,1.0,NaN,google,decline
27669,3516628,0.981352,15.247,C,1,0.0,12,0.913103,1.0,lenovo,google,decline
22702,3511661,0.979909,53.856,C,1,0.0,1,1.000000,1.0,NaN,google,decline


In [4]:
def profile(df, label):
    return pd.Series({
        "n": len(df), "first txn on account": df.is_first_txn_uid.mean(), "has identity": df.has_identity.mean(),
        "median amount": df.TransactionAmt.median(), "card used in last hour": (df.card_tx_1h > 0).mean(),
        "ProductCD=W": (df.ProductCD == "W").mean(), "ProductCD=C": (df.ProductCD == "C").mean(),
    }, name=label)

legit = test[test.isFraud == 0]
top_legit = legit[legit[p] >= legit[p].quantile(0.99)]
pd.concat([profile(legit, "all legitimate"), profile(top_legit, "top 1% scored legitimate"),
           profile(test[test.isFraud == 1], "all fraud")], axis=1).round(3)

,all legitimate,top 1% scored legitimate,all fraud
n,85498.000,855.000,3083.000
first txn on account,0.297,0.257,0.278
has identity,0.198,0.612,0.572
median amount,68.530,76.543,63.950
card used in last hour,0.250,0.434,0.377
ProductCD=W,0.797,0.363,0.419
ProductCD=C,0.095,0.473,0.412


**Where friction lands (DRAFT):** _the false positives skew toward ... (new accounts? no identity? product C?)._

## 2. Fraud the recommended policy approved

In [5]:
missed = test[(test.isFraud == 1) & (test.decision == "approve")]
caught = test[(test.isFraud == 1) & (test.decision != "approve")]
print(f"missed {len(missed)} of {int(test.isFraud.sum())} frauds; ${missed.TransactionAmt.sum():,.0f} of ${test.TransactionAmt[test.isFraud == 1].sum():,.0f}")
pd.concat([missed.amount_band.value_counts(normalize=True).rename("missed"),
           caught.amount_band.value_counts(normalize=True).rename("caught")], axis=1).round(3)

missed 1468 of 3083 frauds; $206,593 of $469,609


,missed,caught
amount_band,,
$50-200,0.486,0.334
$10-50,0.311,0.402
$200-1k,0.156,0.186
<$10,0.031,0.064
$1k+,0.015,0.013


In [6]:
pd.concat([missed.ProductCD.value_counts(normalize=True).rename("missed"),
           caught.ProductCD.value_counts(normalize=True).rename("caught")], axis=1).round(3)

,missed,caught
ProductCD,,
W,0.660,0.201
C,0.210,0.596
S,0.056,0.038
H,0.046,0.060
R,0.029,0.105


In [7]:
# card-testing check: small amounts on a card already busy in the last hour
for name, d in (("missed", missed), ("caught", caught)):
    small = d.TransactionAmt < 10
    print(f"{name:7s} small(<$10): {small.mean():.1%}   small & card used in last hour: {(small & (d.card_tx_1h > 0)).mean():.1%}")

missed  small(<$10): 2.6%   small & card used in last hour: 1.4%
caught  small(<$10): 6.4%   small & card used in last hour: 3.8%


**Missed-fraud pattern (DRAFT):** _most missed fraud is small-dollar ..., which the cost policy approves on purpose: declining a $5 transaction risks $5 of friction to save ~$20._

## 3. Cold start: accounts never seen before the test window

In [8]:
rows = []
for k, v in res["experiments"].items():
    if "test_new_uid" in v:
        rows.append({"model": k, "all test PR-AUC": v["test"]["pr_auc"], "new-account PR-AUC": v["test_new_uid"]["pr_auc"],
                     "new-account base rate": v["test_new_uid"]["base_rate"]})
print(f"{res['new_uid_test_share']:.1%} of test rows are new accounts")
pd.DataFrame(rows).round(4)

54.6% of test rows are new accounts


,model,all test PR-AUC,new-account PR-AUC,new-account base rate
0,lgbm,0.5182,0.5574,0.0476
1,lgbm_spw,0.5138,0.5451,0.0476
2,lgbm_tuned,0.5271,0.5565,0.0476
3,abl_raw_text,0.5223,0.5590,0.0476
4,abl_no_behavior,0.5222,0.5634,0.0476
5,abl_interpretable,0.2251,0.2396,0.0476
6,rules,0.0350,0.0509,0.0476
7,lr,0.0976,0.1509,0.0476
8,lgbm_cost,0.5077,0.5423,0.0476


## 4. Reason codes for example transactions

In [9]:
ex = load_json(paths.reports / "explain.json")["examples"]
for e in ex:
    print(f"{e['type']:15s} id={e['TransactionID']}  p={e['p']:.3f}")
    for r in e["reasons"]:
        print("    -", r["text"])

caught_fraud    id=3545361  p=0.999
    - masked vendor risk signals (V/C/D/M/id columns)
    - transaction amount (USD): $300.00
    - recipient email provider (normalized): google
caught_fraud    id=3519815  p=0.998
    - masked vendor risk signals (V/C/D/M/id columns)
    - transaction amount (USD): $175.00
    - recipient email provider (normalized): google
caught_fraud    id=3519799  p=0.998
    - masked vendor risk signals (V/C/D/M/id columns)
    - transaction amount (USD): $175.00
    - recipient email provider (normalized): google
false_positive  id=3568818  p=0.989
    - masked vendor risk signals (V/C/D/M/id columns)
    - number of earlier transactions on this card key: 2,605
    - card3: 185
false_positive  id=3569749  p=0.987
    - masked vendor risk signals (V/C/D/M/id columns)
    - recipient email provider (normalized): google
    - card type (debit/credit): credit
false_positive  id=3516628  p=0.981
    - masked vendor risk signals (V/C/D/M/id columns)
    - device fa